In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import f1_score, recall_score, precision_score

train_df = pd.read_csv('train_features.csv')
test_df = pd.read_csv('test_features.csv')

X_train = train_df.iloc[:, :-1].values 
y_train = train_df.iloc[:, -1].values
X_test = test_df.iloc[:, :-1].values
y_test = test_df.iloc[:, -1].values


scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


kernel_configs = {
    'linear':  dict(kernel='linear', C=1.0),
    'poly':    dict(kernel='poly', C=0.1, degree=2),
    'rbf':     dict(kernel='rbf', C=15.0, gamma=0.05), 
    'sigmoid': dict(kernel='sigmoid', C=1.0, gamma=0.005)
}

results = {}
models = {}

print("--- Multi-Kernel Analysis: PneumoniaMNIST (128 Features) ---")
for knl, params in kernel_configs.items():
    model = SVC(**params, probability=True)
    model.fit(X_train_scaled, y_train)
    models[knl] = model

    
    probs = model.predict_proba(X_test_scaled)[:, 1]
    threshold = 0.5
    preds = (probs >= threshold).astype(int)

    results[knl] = {
        'p': precision_score(y_test, preds),
        'r': recall_score(y_test, preds),
        'f1': f1_score(y_test, preds)
    }

    m = results[knl]
    print(f"{knl.upper():<10} | F1: {m['f1']:.4f} | Precision: {m['p']:.4f} | Recall: {m['r']:.4f}")

rbf_model = models['rbf']
sample_idx = 0 

x_sample = X_test_scaled[sample_idx]
y_sample = y_test[sample_idx]
decision_score = rbf_model.decision_function([x_sample])

np.savetxt("support_vectors_rbf.txt", rbf_model.support_vectors_)
np.savetxt("dual_coeff_rbf.txt", rbf_model.dual_coef_)
np.savetxt("intercept_rbf.txt", rbf_model.intercept_)

np.savetxt("xtest_rbf.txt", x_sample)
np.savetxt("ytest_rbf.txt", [y_sample])
np.savetxt("yclassificationscore.txt", decision_score)

print(f"\n[System] Best Kernel found: {max(results, key=lambda k: results[k]['f1']).upper()}")
print("[System] RBF parameters and test sample saved successfully for FHE validation.")

--- Multi-Kernel Analysis: PneumoniaMNIST (128 Features) ---
LINEAR     | F1: 0.8831 | Precision: 0.8487 | Recall: 0.9205
POLY       | F1: 0.8921 | Precision: 0.8460 | Recall: 0.9436
RBF        | F1: 0.8976 | Precision: 0.8558 | Recall: 0.9436
SIGMOID    | F1: 0.8859 | Precision: 0.8582 | Recall: 0.9154

[System] Best Kernel found: RBF
[System] RBF parameters and test sample saved successfully for FHE validation.
